# D3 stronger MIA — DP-aware shadow membership inference

Standard A5 (experiment 08) ran the Shokri attack with undefended EEGNet shadows against an undefended target — MIA AUC = 0.878. The standard published attack against DP-trained models is to also run the shadows in the same defended pipeline (Yeom 2018; Shokri 2017): the attacker's shadow distribution matches the defender's noise distribution, which uniformly strengthens the attack.

This notebook runs that DP-aware attack against the DP-SGD ε=3 victim. 8 shadows, 30 epochs each, on the 104 PhysioNet subjects.

**Runtime → L4 GPU.** Expected wall: ~3-4 hours (9 DP-SGD trainings × ~20 min each + attack fit).

**Don't `Save a copy in GitHub` from Colab.**

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich scipy

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.27_d3_membership_aware_attacker --all

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
RESULTS = 'results/27_d3_membership_aware_attacker.json'
TAG = 'd3_dp_aware_mia'
try:
    git_sha = subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
except Exception: git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': 'experiments.27_d3_membership_aware_attacker',
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': [RESULTS]}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json','w') as f: json.dump(meta, f, indent=2)
if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing; re-run the experiment cell.')
print('--- BEGIN 27_d3_membership_aware_attacker.json ---')
print(open(RESULTS).read())
print('--- END 27_d3_membership_aware_attacker.json ---')
print()
print('--- BEGIN run meta ---')
print(json.dumps(meta, indent=2))
print('--- END run meta ---')